# Data Loading

In [3]:
import pandas as pd
import re
#import nltk

import spacy

# Import VADER model
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
sia = SentimentIntensityAnalyzer()

# Import Doc for extending the SpaCy pipeline
from spacy.tokens import Doc

In [4]:
games_df = pd.read_csv('games.csv')
reviews_df = pd.read_csv('dataset.csv')
#reviews_df.head()

games_df.head()
#reviews_df.head()

,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,About the game,Supported languages,...,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,NaN,[],...,0,0,0,NaN,NaN,NaN,NaN,NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,"Springtime, April: when the cherry trees come ...",['English'],...,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,...",Adventure,"Adventure,Visual Novel,Anime,Cute",https://shared.akamai.steamstatic.com/store_it...,NaN
1034400,Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,0,4.99,0,0,"Immerse yourself in the most beloved, mystical...","['English', 'French', 'German', 'Russian']",...,0,0,0,Somer Games,8floor,"Single-player,Family Sharing",Casual,"Casual,Card Game,Solitaire,Puzzle,Hidden Objec...",https://shared.akamai.steamstatic.com/store_it...,NaN
3292190,버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,1,0,8.99,0,1,"synopsis 'Hello, I'm Hiyoro, a new YouTuber!' ...",['Korean'],...,0,0,0,유진게임즈,유진게임즈,"Single-player,Steam Achievements,Family Sharing","Casual,Indie,Simulation",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN
3631080,Maze Quest VR,"Apr 24, 2025",0 - 20000,0,0,4.99,0,0,Its not just a Maze; its a Quest! Enter the ca...,['English'],...,0,0,0,Reality Expanded LLC,Reality Expanded LLC,"Single-player,VR Only,Steam Leaderboards,Famil...","Action,Early Access",NaN,https://shared.akamai.steamstatic.com/store_it...,NaN


In [5]:
listed_genres = set()
for genres in games_df['Genres'].unique().tolist():
    for genre in str(genres).split(','):
        listed_genres.add(genre)


listed_tags = set()
for tags in games_df['Tags'].unique().tolist():
    for tag in str(tags).split(','):
        listed_tags.add(tag)

print(listed_genres)
print(len(listed_genres))

print(listed_tags)
print(len(listed_tags))

{'RPG', 'Short', 'Adventure', 'Movie', 'Strategy', 'Nudity', 'Software Training', 'Simulation', 'Web Publishing', 'Sexual Content', 'Education', 'Free To Play', 'Tutorial', 'Photo Editing', 'Gore', 'Video Production', 'Racing', 'nan', 'Episodic', 'Animation & Modeling', 'Sports', 'Audio Production', 'Early Access', '360 Video', 'Accounting', 'Violent', 'Design & Illustration', 'Massively Multiplayer', 'Action', 'Indie', 'Game Development', 'Documentary', 'Utilities', 'Casual'}
34
{'Surreal', 'Outbreak Sim', 'Driving', 'RPG', 'Vikings', 'Co-op', 'Martial Arts', 'Female Protagonist', 'Local Co-Op', 'Trains', 'Superhero', 'Metroidvania', 'Life Sim', 'Electronic Music', 'Agriculture', 'Roguevania', 'Cute', 'Online Co-Op', 'Bullet Time', 'Political', 'Real-Time with Pause', 'Building', 'Dungeon Crawler', 'Dark Humor', 'Real Time Tactics', 'Singleplayer', 'Ambient', 'Dystopian', 'Sci-fi', 'Family Friendly', 'Hockey', '3D Vision', 'Clicker', 'Modern', 'Cozy', 'Split Screen', 'Hand-drawn', 'Es

In [6]:
narrative_games = games_df[games_df['Tags'].str.contains('Story Rich') | games_df['Tags'].str.contains('Narrative') | games_df['Tags'].str.contains('Choices Matter')]

narrative_games.head()

,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,About the game,Supported languages,...,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
1108640,OMNIMUS,"Sep 25, 2019",0 - 20000,0,0,0.99,0,1,Omnimus is an interactive visual novel in an u...,"['English', 'Russian']",...,0,0,0,Angry games,Total,"Single-player,Family Sharing","Adventure,Indie","Adventure,Indie,Visual Novel,Interactive Ficti...",https://shared.akamai.steamstatic.com/store_it...,NaN
1514380,COGEN: Sword of Rewind / COGEN: 大鳥こはくと刻の剣,"Jan 26, 2022",20000 - 50000,0,0,12.49,0,3,■Defy Death by Rewinding Time! Any hit you tak...,"['English', 'Simplified Chinese', 'Traditional...",...,0,0,0,"Gemdrops, Inc.","Gemdrops, Inc.","Single-player,Steam Achievements,Full controll...",Action,"Action-Adventure,Pixel Graphics,Time Manipulat...",https://shared.akamai.steamstatic.com/store_it...,NaN
1479370,Pixel Game Maker Series Werewolf Princess Kaguya,"Dec 20, 2020",0 - 20000,0,0,9.99,0,0,A stylish &amp; exciting action game! Journey ...,"['English', 'Simplified Chinese', 'Traditional...",...,0,0,0,SERIALGAMES Inc.,Gotcha Gotcha Games,"Single-player,Full controller support,Family S...","Action,Indie","Action,Arcade,Character Action Game,2D Platfor...",https://shared.akamai.steamstatic.com/store_it...,NaN
3098390,Spookstopia: Atomic Division,"Jul 31, 2024",0 - 20000,0,0,2.99,0,0,What is possible between the dimensions of Hel...,"['English', 'Russian']",...,0,0,0,TexFrendza,TexFrendza,"Single-player,Family Sharing","Indie,Early Access","Adventure,Platformer,2D Platformer,2D,Mystery,...",https://shared.akamai.steamstatic.com/store_it...,NaN
3215680,Who is Paul,"Oct 21, 2024",0 - 20000,0,0,0.89,70,0,WARNING: The game's content includes sudden lo...,"['English', 'French', 'Italian', 'German', 'Sp...",...,0,0,0,Atlas Sf,Atlas Sf,"Single-player,Steam Achievements,Family Sharing","Action,Adventure,Indie","Action,Adventure,FPS,Dark,Story Rich,Stealth,A...",https://shared.akamai.steamstatic.com/store_it...,NaN


In [7]:
narrative_game_names = narrative_games["AppID"].unique().tolist()

filtered_reviews = reviews_df[reviews_df["app_name"].isin(narrative_game_names)]

print(len(filtered_reviews))
print(len(filtered_reviews["app_name"].unique().tolist()))

filtered_reviews.head()


1236465
1101


,app_id,app_name,review_text,review_score,review_votes
46177,104600,Portal 2 - The Final Hours,"First of all, this is no game but a virtual in...",1,1
46178,104600,Portal 2 - The Final Hours,The Final Hours is not a game (though can't yo...,1,1
46179,104600,Portal 2 - The Final Hours,Get this if you are a geek about how games are...,1,1
46180,104600,Portal 2 - The Final Hours,IT'S NOT A GAME! But it's a very interesting r...,1,1
46181,104600,Portal 2 - The Final Hours,Everything you would want to know about Portal...,1,1


In [8]:
# Cleaning up narrative games by columns that are relevant to us
narrative_games_cleaned = narrative_games[['AppID', 'Name', 'Estimated owners', 'Price', 'About the game', 'Positive', 'Negative', 'Score rank', 'Recommendations', 'Categories', 'Genres', 'Tags']]

review_count = []
pos_ratio = []

for i in range (len(narrative_games_cleaned)):
    game = narrative_games_cleaned.iloc[i]
    game_review_count = game.get('Positive') + game.get('Negative')
    game_pos_ratio = game.get('Positive')/game_review_count

    review_count.append(game_review_count)
    pos_ratio.append(game_pos_ratio)


narrative_games_cleaned['review_count'] = review_count
narrative_games_cleaned['pos_ratio'] = pos_ratio

narrative_games_cleaned.head()

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_23080\642534715.py:10: RuntimeWarning: invalid value encountered in scalar divide
  game_pos_ratio = game.get('Positive')/game_review_count


,AppID,Name,Estimated owners,Price,About the game,Positive,Negative,Score rank,Recommendations,Categories,Genres,Tags,review_count,pos_ratio
1108640,OMNIMUS,"Sep 25, 2019",0,0,Omnimus is an interactive visual novel in an u...,47,6,NaN,0,"Single-player,Family Sharing","Adventure,Indie","Adventure,Indie,Visual Novel,Interactive Ficti...",53,0.886792
1514380,COGEN: Sword of Rewind / COGEN: 大鳥こはくと刻の剣,"Jan 26, 2022",0,0,■Defy Death by Rewinding Time! Any hit you tak...,75,34,NaN,111,"Single-player,Steam Achievements,Full controll...",Action,"Action-Adventure,Pixel Graphics,Time Manipulat...",109,0.688073
1479370,Pixel Game Maker Series Werewolf Princess Kaguya,"Dec 20, 2020",0,0,A stylish &amp; exciting action game! Journey ...,12,0,NaN,0,"Single-player,Full controller support,Family S...","Action,Indie","Action,Arcade,Character Action Game,2D Platfor...",12,1.000000
3098390,Spookstopia: Atomic Division,"Jul 31, 2024",0,0,What is possible between the dimensions of Hel...,1,1,NaN,0,"Single-player,Family Sharing","Indie,Early Access","Adventure,Platformer,2D Platformer,2D,Mystery,...",2,0.500000
3215680,Who is Paul,"Oct 21, 2024",0,70,WARNING: The game's content includes sudden lo...,13,0,NaN,0,"Single-player,Steam Achievements,Family Sharing","Action,Adventure,Indie","Action,Adventure,FPS,Dark,Story Rich,Stealth,A...",13,1.000000


## Data Cleaning

In [9]:
#filters for ascii characters
filter_ascii = filtered_reviews["review_text"].replace(to_replace=r'[^\x00-\x7f]', value='', regex=True)

#removes new lines
remove_newline = filter_ascii.replace(to_replace=r'\d', value='', regex=True)

#removes certain characters used in reviews
remove_otherchars = remove_newline.replace(to_replace=r'[,()*/"1234567890+|]', value='', regex=True)

#removes punctuations
replace_punctuations = remove_otherchars.replace(to_replace=r'[.!?]+', value = ' | ', regex=True)

#remove dashes not used to hyphenate words
remove_nonhyphen_dashes = replace_punctuations.replace(to_replace=r'- | -', value=' ', regex=True)

#remove extra whitespace
remove_extra_whitespace = remove_nonhyphen_dashes.replace(to_replace=r' +', value=' ', regex=True)

# assign cleaned entries back to the data frame
new = remove_extra_whitespace
filtered_reviews['review_text'] = new

# apply lowercasing
filtered_reviews['review_text'] = filtered_reviews['review_text'].str.lower()
filtered_reviews

print(filtered_reviews)

         app_id                    app_name  \
46177    104600  Portal 2 - The Final Hours   
46178    104600  Portal 2 - The Final Hours   
46179    104600  Portal 2 - The Final Hours   
46180    104600  Portal 2 - The Final Hours   
46181    104600  Portal 2 - The Final Hours   
...         ...                         ...   
6404109    9880            Champions Online   
6404110    9880            Champions Online   
6404111    9880            Champions Online   
6404112    9880            Champions Online   
6404113    9880            Champions Online   

                                               review_text  review_score  \
46177    first of all this is no game but a virtual int...             1   
46178    the final hours is not a game though can't you...             1   
46179    get this if you are a geek about how games are...             1   
46180    it's not a game | but it's a very interesting ...             1   
46181    everything you would want to know about portal

In [12]:
## additional cleaning based on Kaggle User Ali Marjani, PhD
## via https://www.kaggle.com/code/dardodel/steam-reviews-auto-topic-modeling-w-transformers/notebook

print("before removing duplicates: " + str(filtered_reviews.shape))

# drop duplicates
## NOTE: only this actually decreased stuff lmao
filtered_reviews.drop_duplicates(['review_text', 'review_score'], inplace = True)
print("after removing duplicates: " + str(filtered_reviews.shape))


before removing duplicates: (1017234, 6)
after removing duplicates: (1017234, 6)


In [13]:
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

word_tokens = [ (str(review)).split() for review in filtered_reviews['review_text']]

stop_words = set(stopwords.words('english'))
stop_words.add('us')

lemmatized_reviews = []
for tokens in word_tokens:
    lemmatized_reviews.append([lemmatizer.lemmatize(token) for token in tokens if token not in stop_words or token=='not'])

lemmatized_reviews = [" ".join(review) for review in lemmatized_reviews]

filtered_reviews["lemmatized_reviews"] = lemmatized_reviews
filtered_reviews

,app_id,app_name,review_text,review_score,review_votes,lemmatized_reviews
46177,104600,Portal 2 - The Final Hours,first of all this is no game but a virtual int...,1,1,first game virtual interactive book good one m...
46178,104600,Portal 2 - The Final Hours,the final hours is not a game though can't you...,1,1,final hour not game though can't imagine one s...
46179,104600,Portal 2 - The Final Hours,get this if you are a geek about how games are...,1,1,get geek game made |
46180,104600,Portal 2 - The Final Hours,it's not a game | but it's a very interesting ...,1,1,not game | interesting reading interested crea...
46181,104600,Portal 2 - The Final Hours,everything you would want to know about portal...,1,1,everything would want know portal right intera...
...,...,...,...,...,...,...
6404107,9880,Champions Online,i paid for a month subscription to check out a...,-1,1,paid month subscription check much full game c...
6404108,9880,Champions Online,my absolute favorite superhero games even thou...,-1,0,absolute favorite superhero game even though g...
6404109,9880,Champions Online,the gane starts to get boreing when you reatch...,-1,0,gane start get boreing reatch level know every...
6404110,9880,Champions Online,best character creator i've seen | edit: they ...,-1,0,best character creator seen | edit: removed it...


In [14]:
narrative_games_available = narrative_games_cleaned[narrative_games_cleaned["AppID"].isin(filtered_reviews["app_name"].unique())]

narrative_games_available.head()

,AppID,Name,Estimated owners,Price,About the game,Positive,Negative,Score rank,Recommendations,Categories,Genres,Tags,review_count,pos_ratio
558660,it's always monday,"Nov 22, 2016",0,0,it's always monday- is a first person explorat...,228,61,NaN,0,"Single-player,Family Sharing","Adventure,Indie,Simulation","Adventure,Indie,Simulation,First-Person,Psycho...",289,0.788927
242700,Injustice: Gods Among Us Ultimate Edition,"Nov 12, 2013",69,0,Injustice: Gods Among Us Ultimate Edition enha...,16534,2651,NaN,9627,"Single-player,Multi-player,Steam Achievements,...",Action,"Fighting,Superhero,Action,Multiplayer,Comic Bo...",19185,0.861819
277470,The Book of Legends,"Mar 14, 2014",0,0,"The Demon of Fear, Azutura, is slowly but sure...",191,94,NaN,139,"Single-player,Steam Achievements,Full controll...","Adventure,Indie,RPG","RPG,RPGMaker,Indie,Adventure,Anime,JRPG,Single...",285,0.670175
222480,Resident Evil Revelations,"May 20, 2013",101,75,Resident Evil ® Revelations returns redefined ...,12258,2947,NaN,10007,"Single-player,Multi-player,Co-op,Steam Achieve...","Action,Adventure","Horror,Action,Zombies,Survival Horror,Third-Pe...",15205,0.806182
311010,The Way,"May 20, 2016",0,94,Journey to another planet and discover its sec...,1053,212,NaN,883,"Single-player,Steam Achievements,Steam Trading...","Action,Adventure,Indie","Adventure,Indie,Action,Pixel Graphics,Platform...",1265,0.832411


In [15]:
# Getting highest rating based on number of reviews and number of positive reviews
top5_df = narrative_games_available.nlargest(5, ["review_count", "pos_ratio"], 'first')
top5_games = []

for i in range (len(top5_df)):
    game = top5_df.iloc[i]
    top5_games.append(game.get("AppID")) #Get all the game names
    

top5_df.head()

,AppID,Name,Estimated owners,Price,About the game,Positive,Negative,Score rank,Recommendations,Categories,Genres,Tags,review_count,pos_ratio
292030,The Witcher 3: Wild Hunt,"May 18, 2015",15893,90,NaN,802993,32356,NaN,797928,"Single-player,Steam Achievements,Steam Trading...",RPG,"Open World,RPG,Story Rich,Atmospheric,Mature,F...",835349,0.961266
239140,Dying Light,"Jan 26, 2015",4953,85,"IMPORTANT: Enjoy enhanced visuals, a remastere...",435687,21945,NaN,361016,"Single-player,Multi-player,PvP,Online PvP,Co-o...","Action,RPG","Zombies,Survival Horror,Horror,Online Co-Op,Op...",457632,0.952047
620,Portal 2,"Apr 18, 2011",1959,80,Portal 2 draws from the award-winning formula ...,427835,5675,NaN,374061,"Single-player,Multi-player,Co-op,Online Co-op,...","Action,Adventure","Platformer,Puzzle,First-Person,Dark Humor,Stor...",433510,0.986909
374320,DARK SOULS™ III,"Apr 11, 2016",4381,0,Get the DARK SOULS™ III Season Pass now and ch...,390326,23449,NaN,269868,"Single-player,Multi-player,Co-op,Steam Achieve...",Action,"Souls-like,Dark Fantasy,Difficult,RPG,Atmosphe...",413775,0.943329
377160,Fallout 4,"Nov 9, 2015",11179,60,"From Bethesda Game Studios, the award-winning ...",326054,66233,NaN,276492,"Single-player,Steam Achievements,Full controll...",RPG,"Open World,Post-apocalyptic,Singleplayer,Explo...",392287,0.831162


In [16]:
# Did it this way since games with no reviews will register as 0 and will clog up the space for lowest rated games
# We still want a significant amount of reviews to base what is actually bad vs what is underrepresented/underreviewed
cond = narrative_games_available.get("review_count") >= 5000
temp_bot_df = narrative_games_available.where(cond)
bot5_df = temp_bot_df.nsmallest(5, "pos_ratio", 'first')

bot5_games = []

for i in range (len(bot5_df)):
    game = bot5_df.iloc[i]
    bot5_games.append(game.get("AppID")) #Get all the game names

bot5_df.head()
#print(bot5_games)

,AppID,Name,Estimated owners,Price,About the game,Positive,Negative,Score rank,Recommendations,Categories,Genres,Tags,review_count,pos_ratio
360430,Mafia III: Definitive Edition,"May 19, 2020",869.0,80.0,Mafia III: Definitive Edition includes the mai...,32366.0,24142.0,NaN,38656.0,"Single-player,Steam Achievements,Full controll...","Action,Adventure","Crime,Open World,Action,Story Rich,Singleplaye...",56508.0,0.572768
55100,Homefront,"Mar 14, 2011",12.0,90.0,The year is 2027. The world as we know it is u...,7787.0,4980.0,NaN,3864.0,"Single-player,Multi-player,Steam Achievements,...",Action,"Action,FPS,Shooter,Multiplayer,Singleplayer,Fi...",12767.0,0.609932
495910,Dungeon Fighter Online,"Aug 9, 2016",173.0,0.0,With fast-paced gameplay and unique side scrol...,3397.0,2141.0,NaN,0.0,"Multi-player,MMO,PvP,Online PvP,Steam Achievem...","Action,Adventure,Massively Multiplayer,RPG,Fre...","Free to Play,Anime,RPG,Character Customization...",5538.0,0.613398
216150,MapleStory,"Aug 9, 2012",8092.0,0.0,Join over 260 Million Global Players in MapleS...,8882.0,5305.0,NaN,0.0,"Single-player,Multi-player,MMO,Co-op,Steam Tra...","Action,Adventure,Casual,Massively Multiplayer,...","MMORPG,2D Platformer,Character Customization,L...",14187.0,0.626066
32500,STAR WARS™: The Force Unleashed™ II,"Oct 26, 2010",43.0,50.0,The Star Wars® Saga continues with Star Wars®:...,4692.0,2696.0,NaN,5544.0,"Single-player,Partial Controller Support,Steam...",Action,"Action,Singleplayer,Hack and Slash,Third Perso...",7388.0,0.635084


In [17]:
#Get reviews from top5 games in filtered reviews
sampled_top5_array = []

for i in range(len(top5_games)):
    temp_df=filtered_reviews[filtered_reviews["app_name"].str.contains(top5_games[i])]
    sampled_temp_df=temp_df.sample(1000)

    #sample reviews from each
    sampled_top5_array.append(sampled_temp_df)

sampled_top5_df = pd.concat(sampled_top5_array)
sampled_top5_df.head()

,app_id,app_name,review_text,review_score,review_votes,lemmatized_reviews
3654158,292030,The Witcher 3: Wild Hunt,honestly one of the best games i've ever playe...,1,0,honestly one best game ever played loved every...
3649073,292030,The Witcher 3: Wild Hunt,the witcher is a astounding game in every aspe...,1,0,witcher astounding game every aspect including...
3646301,292030,The Witcher 3: Wild Hunt,best game i ever played | bought it twice | th...,1,0,best game ever played | bought twice | thanks ...
3632247,292030,The Witcher 3: Wild Hunt,like skyrim but for adults with good graphics,1,1,like skyrim adult good graphic
3644067,292030,The Witcher 3: Wild Hunt,this game is mind blowing | over hours of game...,1,0,game mind blowing | hour gameplay still not do...


In [18]:
#Get reviews from bottom5 games in filtered reviews
sampled_bot5_array = []

for i in range(len(bot5_games)):
    temp_df=filtered_reviews[filtered_reviews["app_name"].str.contains(bot5_games[i])]
    if len(temp_df) > 1000:
        sampled_temp_df=temp_df.sample(1000)

        #sample reviews from each
        sampled_bot5_array.append(sampled_temp_df)
    else:
        sampled_bot5_array.append(temp_df)

sampled_bot5_df = pd.concat(sampled_bot5_array)
sampled_bot5_df.head()

,app_id,app_name,review_text,review_score,review_votes,lemmatized_reviews
4694857,360430,Mafia III: Definitive Edition,this game is wait for it | | fps: you all know...,-1,0,game wait | | fps: know locked fps going relea...
4695830,360430,Mafia III: Definitive Edition,the most pisspoor excuse for a mafia game i ab...,-1,0,pisspoor excuse mafia game absolutely loved ma...
4694996,360430,Mafia III: Definitive Edition,i-k turbo boosted gb ram gtx ti ftw p max sett...,-1,1,i-k turbo boosted gb ram gtx ti ftw p max sett...
4695237,360430,Mafia III: Definitive Edition,other than the obvious fps problems it looks l...,-1,0,obvious fps problem look like someone's smeare...
4695966,360430,Mafia III: Definitive Edition,yaya blah blah i just played a couple of minut...,-1,1,yaya blah blah played couple minute | buy game...


In [24]:
sampled_reviews = pd.concat([sampled_top5_df, sampled_bot5_df])

sampled_reviews.head()

,app_id,app_name,review_text,review_score,review_votes,lemmatized_reviews
3654158,292030,The Witcher 3: Wild Hunt,honestly one of the best games i've ever playe...,1,0,honestly one best game ever played loved every...
3649073,292030,The Witcher 3: Wild Hunt,the witcher is a astounding game in every aspe...,1,0,witcher astounding game every aspect including...
3646301,292030,The Witcher 3: Wild Hunt,best game i ever played | bought it twice | th...,1,0,best game ever played | bought twice | thanks ...
3632247,292030,The Witcher 3: Wild Hunt,like skyrim but for adults with good graphics,1,1,like skyrim adult good graphic
3644067,292030,The Witcher 3: Wild Hunt,this game is mind blowing | over hours of game...,1,0,game mind blowing | hour gameplay still not do...


In [20]:
## Review score of -1: would not recommend

would_recommend = filtered_reviews[filtered_reviews["review_score"]==-1]
print(len(would_recommend))
print(len(would_recommend["app_name"].unique().tolist()))

would_recommend.head()

130621
1029


,app_id,app_name,review_text,review_score,review_votes,lemmatized_reviews
46219,104600,Portal 2 - The Final Hours,i'm sorry to say this glorified pdf file is of...,-1,1,sorry say glorified pdf file little interest a...
46276,104600,Portal 2 - The Final Hours,this game sorry digital book is a rip off | yo...,-1,1,game sorry digital book rip | not paying speci...
46279,104600,Portal 2 - The Final Hours,for those of you who might think this is a gam...,-1,1,might think game not | got little interested d...
46281,104600,Portal 2 - The Final Hours,don't buy this | no seriously this is just an ...,-1,0,buy | seriously article nice package | even vi...
46282,104600,Portal 2 - The Final Hours,unfortunetly every time i go to do something o...,-1,1,unfortunetly every time go something gamebehin...


# Exploratory Data Analysis

In [21]:
#inserting data from 1 dataset to the next
estimated_owners = []
prices = []
last_game = ""
estimate = ""
price = 0
story_rich=[]
narrative=[]
choices_matter=[]
to_append = [False, False, False]
for i in range (len(filtered_reviews)):
    game = filtered_reviews.iloc[i,1] #sometimes its 1, sometimes its 2,,, idk why 
    if game != last_game:
        estimate = narrative_games.loc[narrative_games["AppID"]==game, "Release date"].iloc[0]
        #for some reason realease date is the header for estimated owners dont ask me
        game_tags = narrative_games.loc[narrative_games["AppID"]==game, "Tags"].iloc[0]
        to_append[0] = "Story Rich" in game_tags
        to_append[1] = "Narrative" in game_tags
        to_append[2] = "Choices Matter" in game_tags
        #same here,, required age = price
        price = narrative_games.loc[narrative_games["AppID"]==game, "Required age"].iloc[0]

    estimated_owners.append(estimate)
    story_rich.append(to_append[0])
    narrative.append(to_append[1])
    choices_matter.append(to_append[2])
    prices.append(price)
    last_game = game    


filtered_reviews["estimated_owners"] = estimated_owners 
filtered_reviews["story_rich"] = story_rich
filtered_reviews["narrative"] = narrative 
filtered_reviews["choices_matter"] = choices_matter 
filtered_reviews["price"] = prices

estimated_owners = []
prices = []
last_game = ""
estimate = ""
price = 0
story_rich=[]
narrative=[]
choices_matter=[]
filtered_reviews.head()

,app_id,app_name,review_text,review_score,review_votes,lemmatized_reviews,estimated_owners,story_rich,narrative,choices_matter,price
46177,104600,Portal 2 - The Final Hours,first of all this is no game but a virtual int...,1,1,first game virtual interactive book good one m...,200000 - 500000,True,False,False,0.79
46178,104600,Portal 2 - The Final Hours,the final hours is not a game though can't you...,1,1,final hour not game though can't imagine one s...,200000 - 500000,True,False,False,0.79
46179,104600,Portal 2 - The Final Hours,get this if you are a geek about how games are...,1,1,get geek game made |,200000 - 500000,True,False,False,0.79
46180,104600,Portal 2 - The Final Hours,it's not a game | but it's a very interesting ...,1,1,not game | interesting reading interested crea...,200000 - 500000,True,False,False,0.79
46181,104600,Portal 2 - The Final Hours,everything you would want to know about portal...,1,1,everything would want know portal right intera...,200000 - 500000,True,False,False,0.79


### enable gpu

In [22]:
import torch

# Check if a CUDA-capable GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# This line checks if cuda GPU is available
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.14.1+cu126
True
NVIDIA GeForce RTX 3050 6GB Laptop GPU


# Sentiment Analysis

### Without Sampled Data

In [ ]:
from IPython.display import clear_output

sentiment_analysis = lambda doc: sia.polarity_scores(doc.text)
nlp_with_sentiment = spacy.load("en_core_web_sm")
Doc.set_extension("sentiment", getter=sentiment_analysis, force=True)

review_docs = nlp_with_sentiment.pipe(lemmatized_reviews, batch_size=32)
i=0
com_sent = []
pos_sent = []
neg_sent = []
neu_sent = []
for doc in review_docs:
    sentiment = doc._.sentiment
    com_sent.append(sentiment['compound'])
    pos_sent.append(sentiment['pos'])
    neg_sent.append(sentiment['neg'])
    neu_sent.append(sentiment['neu'])
    i+=1
    if(i%1000==0):
        clear_output(wait=True)
        print(str(i)+'/'+str(len(filtered_reviews)))
filtered_reviews['com_sent'] = com_sent
filtered_reviews['pos_sent'] = pos_sent
filtered_reviews['neg_sent'] = neg_sent
filtered_reviews['neu_sent'] = neu_sent

filtered_reviews.head()
filtered_reviews.to_csv("output_senti.csv")

In [46]:

filtered_reviews['pos_sent'] = 0.0
filtered_reviews['neg_sent'] = 0.0

i=0
for index, row in filtered_reviews.iterrows():
    texts = row['lemmatized_reviews']
    temp_neg = []
    temp_neu = []
    temp_pos = []
    for text in texts.split('|'):
        if text.isspace() or len(text)==0:
            continue
        encoded_input = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
        encoded_input = {
            key: value.to(device)
            for key, value in encoded_input.items()
        }
        with torch.no_grad():
            output = model(**encoded_input)
        
        scores = output[0][0].cpu().numpy()
        scores = softmax(scores)
        temp_neg.append(scores[0])
        temp_pos.append(scores[1])

    if(len(temp_neg)==0):
        temp_neg.append(0)
        temp_pos.append(0)
    
    filtered_reviews.at[index, 'pos_sent'] = sum(temp_pos)/len(temp_pos)
    filtered_reviews.at[index, 'neg_sent'] = sum(temp_neg)/len(temp_neg)

    i+=1
    if(i==1028):
        break

filtered_reviews.head()


,app_id,app_name,review_text,review_score,review_votes,lemmatized_reviews,estimated_owners,story_rich,narrative,choices_matter,price,pos_sent,neg_sent
46177,104600,Portal 2 - The Final Hours,first of all this is no game but a virtual int...,1,1,first game virtual interactive book good one m...,200000 - 500000,True,False,False,0.79,0.985086,0.014914
46178,104600,Portal 2 - The Final Hours,the final hours is not a game though can't you...,1,1,final hour not game though can't imagine one s...,200000 - 500000,True,False,False,0.79,0.406472,0.593528
46179,104600,Portal 2 - The Final Hours,get this if you are a geek about how games are...,1,1,get geek game made |,200000 - 500000,True,False,False,0.79,0.991366,0.008634
46180,104600,Portal 2 - The Final Hours,it's not a game | but it's a very interesting ...,1,1,not game | interesting reading interested crea...,200000 - 500000,True,False,False,0.79,0.740863,0.259137
46181,104600,Portal 2 - The Final Hours,everything you would want to know about portal...,1,1,everything would want know portal right intera...,200000 - 500000,True,False,False,0.79,0.903454,0.096546


### With Sampled Data

In [26]:
from IPython.display import clear_output

sentiment_analysis = lambda doc: sia.polarity_scores(doc.text)
nlp_with_sentiment = spacy.load("en_core_web_sm")
Doc.set_extension("sentiment", getter=sentiment_analysis, force=True)

review_docs = nlp_with_sentiment.pipe(sampled_reviews["lemmatized_reviews"], batch_size=32)
i=0
com_sent = []
pos_sent = []
neg_sent = []
neu_sent = []
for doc in review_docs:
    sentiment = doc._.sentiment
    com_sent.append(sentiment['compound'])
    pos_sent.append(sentiment['pos'])
    neg_sent.append(sentiment['neg'])
    neu_sent.append(sentiment['neu'])
    i+=1
    if(i%1000==0):
        clear_output(wait=True)
        print(str(i)+'/'+str(len(sampled_reviews)))

sampled_reviews['com_sent'] = com_sent
sampled_reviews['pos_sent'] = pos_sent
sampled_reviews['neg_sent'] = neg_sent
sampled_reviews['neu_sent'] = neu_sent

sampled_reviews.head()
sampled_reviews.to_csv("output_senti_sampled.csv")

9000/9520


# Topic Modeling

## Auto Topic Modeling with Transofmers
Based on the methodology of Kaggle User Ali Marjani, PhD in the following notebook: https://www.kaggle.com/code/dardodel/steam-reviews-auto-topic-modeling-w-transformers/notebook

In [ ]:
import numpy as np 
import pandas as pd 
import random as rn
import re
import nltk
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3' 

pd.set_option("display.max_rows", 600)
pd.set_option("display.max_columns", 500)
pd.set_option("max_colwidth", 400)

In [ ]:
willrecco_reviews = filtered_reviews[filtered_reviews.review_score==1]
print(len(willrecco_reviews))

## General Topic Modeling

In [ ]:
#General topic modeling
from gensim.corpora.dictionary import Dictionary
from gensim.models.ldamodel import LdaModel
import pyLDAvis, pyLDAvis.gensim_models

In [ ]:
recco_texts = []
recco_texts_sentences = []

notrecco_texts = []
notrecco_texts_sentences = []

###

recco = filtered_reviews[filtered_reviews["review_score"]==1]
recco_lemmatized = [review for review in recco["lemmatized_reviews"]]
print(len(recco_lemmatized))

for i in range(len(recco_lemmatized)):
    for sentence in recco_lemmatized[i].split(' | '):
        if len(sentence.split()) > 5:
            recco_texts_sentences.append([word for word in sentence.split() if word.isalnum() and word not in stop_words])
    if len((re.sub(r' \| ',' ', recco_lemmatized[i])).split())>5:
        recco_texts.append([word for word in (re.sub(r' \| ',' ', recco_lemmatized[i])).split() if word.isalnum() and word not in stop_words])

notrecco = filtered_reviews[filtered_reviews["review_score"]==-1]
notrecco_lemmatized = [review for review in notrecco["lemmatized_reviews"]]
print(len(notrecco_lemmatized))

for i in range(len(notrecco_lemmatized)):
    for sentence in notrecco_lemmatized[i].split(' | '):
        if len(sentence.split()) > 5:
            notrecco_texts_sentences.append([word for word in sentence.split() if word.isalnum() and word not in stop_words])
    if len((re.sub(r' \| ',' ', notrecco_lemmatized[i])).split())>5:
        notrecco_texts.append([word for word in (re.sub(r' \| ',' ', notrecco_lemmatized[i])).split() if word.isalnum() and word not in stop_words])

In [ ]:
print("RECCO TEXTS LEN:     " + str(len(recco_texts)))
print("RECCO SENTS LEN:     " + str(len(recco_texts_sentences)))
print("NOTRECCO TEXTS LEN:  " + str(len(notrecco_texts)))
print("NOTRECCO SENTS LEN:  " + str(len(notrecco_texts_sentences)))

In [ ]:
dictionary = Dictionary(recco_texts + notrecco_texts) # Create a GenSimdictionary
dictionary.filter_extremes(no_below=20, no_above=0.8) #word has to appear in 20 documents,, idk if thats fair
corpus = [dictionary.doc2bow(text) for text in (recco_texts + notrecco_texts)] # Create a GenSim corpus
lda = LdaModel(
    corpus=corpus, 
    id2word=dictionary, 
    num_topics=7
)

pyLDAvis.enable_notebook()
vis = pyLDAvis.gensim_models.prepare(lda, corpus, dictionary,mds='mmds')

pyLDAvis.display(vis)

In [ ]:
dictionary2 = Dictionary(recco_texts_sentences + notrecco_texts_sentences) # Create a GenSimdictionary
dictionary2.filter_extremes(no_below=20, no_above=0.8) #word has to appear in 20 documents,, idk if thats fair
corpus2 = [dictionary2.doc2bow(text) for text in (recco_texts_sentences + notrecco_texts_sentences)] # Create a GenSim corpus
lda2 = LdaModel(
    corpus=corpus2, 
    id2word=dictionary2, 
    num_topics=7
)
    
pyLDAvis.enable_notebook()
vis2 = pyLDAvis.gensim_models.prepare(lda2, corpus2, dictionary2,mds='mmds')

pyLDAvis.display(vis2)

### Topic Modeling Across Recommended / Not Recommended

In [ ]:
dictionaryReccoTexts = Dictionary(recco_texts) # Create a GenSimdictionary
dictionaryReccoTexts.filter_extremes(no_below=20, no_above=0.8) #word has to appear in 20 documents,, idk if thats fair
corpusRT = [dictionaryReccoTexts.doc2bow(text) for text in recco_texts] # Create a GenSim corpus
ldaRT = LdaModel(
    corpus=corpusRT, 
    id2word=dictionaryReccoTexts, 
    num_topics=7
)
    
pyLDAvis.enable_notebook()
visRT = pyLDAvis.gensim_models.prepare(ldaRT, corpusRT, dictionaryReccoTexts,mds='mmds')

pyLDAvis.display(visRT)

In [ ]:
dictionaryReccoSentences = Dictionary(recco_texts_sentences) # Create a GenSimdictionary
dictionaryReccoSentences.filter_extremes(no_below=20, no_above=0.8) #word has to appear in 20 documents,, idk if thats fair
corpusRS = [dictionaryReccoSentences.doc2bow(text) for text in recco_texts_sentences] # Create a GenSim corpus
ldaRS = LdaModel(
    corpus=corpusRS, 
    id2word=dictionaryReccoSentences, 
    num_topics=7
)
    
pyLDAvis.enable_notebook()
visRS = pyLDAvis.gensim_models.prepare(ldaRS, corpusRS, dictionaryReccoSentences,mds='mmds')

pyLDAvis.display(visRS)

In [ ]:
dictionaryNotReccoTexts = Dictionary(notrecco_texts) # Create a GenSimdictionary
dictionaryNotReccoTexts.filter_extremes(no_below=20, no_above=0.8) #word has to appear in 20 documents,, idk if thats fair
corpusNRT = [dictionaryNotReccoTexts.doc2bow(text) for text in notrecco_texts] # Create a GenSim corpus
ldaNRT = LdaModel(
    corpus=corpusNRT, 
    id2word=dictionaryNotReccoTexts, 
    num_topics=7
)
    
pyLDAvis.enable_notebook()
visNRT = pyLDAvis.gensim_models.prepare(ldaNRT, corpusNRT, dictionaryNotReccoTexts,mds='mmds')

pyLDAvis.display(visNRT)